## 0. Load models

In [1]:
from pathlib import Path
import mlflow
import pandas as pd
import numpy as np

# Resolve the project root independently of PyCharm's notebook working directory.
working_dir = Path.cwd().resolve()
if (working_dir / "data").is_dir() and (working_dir / "notebooks").is_dir():
    project_dir = working_dir
elif working_dir.name == "notebooks" and (working_dir.parent / "data").is_dir():
    project_dir = working_dir.parent
else:
    project_dir = next(
        parent for parent in (working_dir, *working_dir.parents)
        if (parent / "data").is_dir() and (parent / "notebooks").is_dir()
    )

models_dir = project_dir / "models"
results_dir = project_dir / "docs" / "results"
tracking_dir = project_dir / ".mlflow"
artifact_dir = tracking_dir / "artifacts"
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
tracking_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
database_path = tracking_dir / "mlflow.db"
if database_path.exists():
    database_path.chmod(0o664)
mlflow.set_tracking_uri(f"sqlite:///{database_path}")
if mlflow.active_run() is not None:
    mlflow.end_run()

encoded_path = project_dir / "data" / "telco_full_encoded.csv"
df_encoded = pd.read_csv(encoded_path)
print(f"Encoded data path: {encoded_path}")
print(f"Encoded data shape: {df_encoded.shape}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


Encoded data path: /Users/falcon/Developer/FIAP-mle/tech-challenge-fase1/data/telco_full_encoded.csv
Encoded data shape: (7043, 40)
MLflow tracking URI: sqlite:////Users/falcon/Developer/FIAP-mle/tech-challenge-fase1/.mlflow/mlflow.db


## 1. Phase 2: Model comparison

    **Objective:** Evaluate the Step 1 Logistic Regression MVP baseline alongside a Random Forest and a scaled `MLPClassifier`, compare all three with stratified cross-validation, select the champion using F1 as the primary metric, and persist the final deliverables.

Both models use the same leakage-free numeric feature matrix and deterministic split. Cross-validation is performed only on the training partition; the test partition is reserved for the final unbiased comparison.

In [2]:
# Train, evaluate, compare, and persist the baseline, Random Forest, and MLP models.
import joblib

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Use the same leakage-free feature matrix for every model, including the Step 1 MVP.
phase2_feature_exclusions = ["CustomerID", "Count", "Country", "State", "target"]
X_phase2 = df_encoded.drop(columns=phase2_feature_exclusions).copy()
y_phase2 = df_encoded["target"].copy()
X_phase2_train, X_phase2_test, y_phase2_train, y_phase2_test = train_test_split(
    X_phase2,
    y_phase2,
    test_size=0.2,
    stratify=y_phase2,
    random_state=42,
)

cv_phase2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_phase2 = {
    "f1": "f1",
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "accuracy": "accuracy",
}

models_phase2 = {
    # Baseline from Step 1: a simple, interpretable linear classifier.
    "Logistic Regression (MVP baseline)": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "MLPClassifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=0.0005,
            batch_size=64,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=42,
        )),
    ]),
}

model_file_names = {
    "Logistic Regression (MVP baseline)": "telco_full_logistic_regression_model.joblib",
    "Random Forest": "telco_random_forest_model.joblib",
    "MLPClassifier": "telco_mlp_model.joblib",
}
model_slugs = {
    "Logistic Regression (MVP baseline)": "full_logistic_regression",
    "Random Forest": "random_forest",
    "MLPClassifier": "mlp",
}

phase2_results = {}
for model_name, estimator in models_phase2.items():
    cv_scores = cross_validate(
        estimator,
        X_phase2_train,
        y_phase2_train,
        cv=cv_phase2,
        scoring=scoring_phase2,
        n_jobs=1,
        return_train_score=False,
    )
    fitted_estimator = clone(estimator)
    fitted_estimator.fit(X_phase2_train, y_phase2_train)
    y_phase2_pred = fitted_estimator.predict(X_phase2_test)
    y_phase2_proba = fitted_estimator.predict_proba(X_phase2_test)[:, 1]

    phase2_results[model_name] = {
        "estimator": fitted_estimator,
        "parameters": fitted_estimator.get_params(deep=True),
        "cv": {
            metric: {
                "mean": float(np.mean(cv_scores[f"test_{metric}"])),
                "std": float(np.std(cv_scores[f"test_{metric}"])),
            }
            for metric in scoring_phase2
        },
        "test": {
            "accuracy": float(accuracy_score(y_phase2_test, y_phase2_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_phase2_test, y_phase2_pred)),
            "precision": float(precision_score(y_phase2_test, y_phase2_pred, zero_division=0)),
            "recall": float(recall_score(y_phase2_test, y_phase2_pred, zero_division=0)),
            "f1": float(f1_score(y_phase2_test, y_phase2_pred, zero_division=0)),
            "roc_auc": float(roc_auc_score(y_phase2_test, y_phase2_proba)),
            "confusion_matrix": confusion_matrix(y_phase2_test, y_phase2_pred).tolist(),
        },
    }

    # Keep one MLflow run per model so the baseline is tracked alongside Phase 2 models.
    experiment_name = f"telco_churn_phase2_{model_slugs[model_name]}"
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name=f"phase2_{model_slugs[model_name]}"):
        mlflow.log_params({
            key: value for key, value in fitted_estimator.get_params(deep=True).items()
            if isinstance(value, (str, int, float, bool, type(None)))
        })
        mlflow.log_param("model_role", "baseline" if model_name.startswith("Logistic") else "candidate")
        mlflow.log_param("cv_strategy", "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)")
        mlflow.log_param("feature_count", X_phase2_train.shape[1])
        mlflow.log_param("train_rows", X_phase2_train.shape[0])
        mlflow.log_param("test_rows", X_phase2_test.shape[0])
        for metric, values in phase2_results[model_name]["cv"].items():
            mlflow.log_metric(f"cv_{metric}_mean", values["mean"])
            mlflow.log_metric(f"cv_{metric}_std", values["std"])
        for metric, value in phase2_results[model_name]["test"].items():
            if metric != "confusion_matrix":
                mlflow.log_metric(f"test_{metric}", value)
        trusted_types = []
        if model_name == "Random Forest":
            trusted_types = ["sklearn.tree._tree.Tree"]
        elif model_name == "MLPClassifier":
            trusted_types = ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"]
        log_kwargs = {"name": "model"}
        if trusted_types:
            log_kwargs["skops_trusted_types"] = trusted_types
        mlflow.sklearn.log_model(fitted_estimator, **log_kwargs)

# F1 is primary; CV ROC AUC and held-out F1 are tie-breakers across all three models.
champion_name = max(
    phase2_results,
    key=lambda name: (
        phase2_results[name]["cv"]["f1"]["mean"],
        phase2_results[name]["cv"]["roc_auc"]["mean"],
        phase2_results[name]["test"]["f1"],
    ),
)
champion_estimator = phase2_results[champion_name]["estimator"]
for model_name, result in phase2_results.items():
    joblib.dump(result["estimator"], models_dir / model_file_names[model_name])
champion_path = models_dir / f"telco_{model_slugs[champion_name]}_champion.joblib"
joblib.dump(champion_estimator, champion_path)

comparison_rows = []
for model_name, result in phase2_results.items():
    comparison_rows.append({
        "Model": model_name,
        "Role": "Baseline" if model_name.startswith("Logistic") else "Candidate",
        "CV F1 mean": result["cv"]["f1"]["mean"],
        "CV F1 std": result["cv"]["f1"]["std"],
        "CV ROC AUC mean": result["cv"]["roc_auc"]["mean"],
        "Test F1": result["test"]["f1"],
        "Test ROC AUC": result["test"]["roc_auc"],
        "Test balanced accuracy": result["test"]["balanced_accuracy"],
        "Test precision": result["test"]["precision"],
        "Test recall": result["test"]["recall"],
    })
comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["CV F1 mean", "CV ROC AUC mean", "Test F1"], ascending=False
).reset_index(drop=True)

print(f"Comparison feature matrix: {X_phase2.shape[0]} rows x {X_phase2.shape[1]} features")
print("\nBaseline and candidate model comparison:")
display(comparison_df.style.format({column: "{:.4f}" for column in comparison_df.columns if column not in {"Model", "Role"}}))
print(f"\nChampion selected by cross-validated F1 across all three models: {champion_name}")
print(f"Champion artifact: {champion_path}")
for model_name, result in phase2_results.items():
    print(f"\n{model_name} confusion matrix (test):")
    print(np.array(result["test"]["confusion_matrix"]))


/Users/falcon/Developer/FIAP-mle/tech-challenge-fase1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/falcon/Developer/FIAP-mle/tech-challenge-fase1/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the dat

2026/11/02 20:13:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/02 20:13:17 INFO mlflow.store.db.utils: Updating database tables


2026/11/02 20:13:17 INFO mlflow.tracking.fluent: Experiment with name 'telco_churn_phase2_full_logistic_regression' does not exist. Creating a new experiment.


2026/11/02 20:13:24 INFO mlflow.tracking.fluent: Experiment with name 'telco_churn_phase2_random_forest' does not exist. Creating a new experiment.


2026/11/02 20:13:30 INFO mlflow.tracking.fluent: Experiment with name 'telco_churn_phase2_mlp' does not exist. Creating a new experiment.


Comparison feature matrix: 7043 rows x 35 features

Baseline and candidate model comparison:


,Model,Role,CV F1 mean,CV F1 std,CV ROC AUC mean,Test F1,Test ROC AUC,Test balanced accuracy,Test precision,Test recall
0,Random Forest,Candidate,0.6466,0.0263,0.8575,0.6315,0.8465,0.7587,0.5628,0.7193
1,MLPClassifier,Candidate,0.6095,0.0398,0.8493,0.5616,0.8411,0.6993,0.6404,0.5000
2,Logistic Regression (MVP baseline),Baseline,0.5367,0.0108,0.8146,0.5659,0.8309,0.7022,0.6346,0.5107



Champion selected by cross-validated F1 across all three models: Random Forest
Champion artifact: /Users/falcon/Developer/FIAP-mle/tech-challenge-fase1/models/telco_random_forest_champion.joblib

Logistic Regression (MVP baseline) confusion matrix (test):
[[925 110]
 [183 191]]

Random Forest confusion matrix (test):
[[826 209]
 [105 269]]

MLPClassifier confusion matrix (test):
[[930 105]
 [187 187]]


In [3]:
# Persist a human-readable comparison report for the baseline and both candidate models.
def format_metric(value):
    return f"{value:.4f}"

report_lines = [
    "# Model Comparison Results",
    "",
    "## Scope",
    "",
    "This experiment compares the Step 1 Logistic Regression MVP baseline with a Random Forest and a scaled scikit-learn `MLPClassifier`. All three models use the same encoded numeric features, stratified 80/20 split, and five-fold stratified cross-validation on the training partition.",
    "",
    "## Evaluation criteria",
    "",
    "F1 score is the primary selection metric because the churn target is imbalanced and both precision and recall matter. CV ROC AUC and held-out test F1 are tie-breakers; balanced accuracy, precision, recall, accuracy, and confusion matrices provide additional context.",
    "",
    f"- Training rows: {len(X_phase2_train)}",
    f"- Test rows: {len(X_phase2_test)}",
    f"- Prediction-time numeric features: {X_phase2.shape[1]}",
    "- Cross-validation: StratifiedKFold, 5 folds, shuffle=True, random_state=42",
    "- Feature exclusions: CustomerID, Count, Country, State, target (derived churn fields are removed during Step 1 preprocessing)",
    "",
    "## Model parameters",
    "",
    "### Logistic Regression (MVP baseline)",
    "",
    "```python",
    "LogisticRegression(random_state=42)",
    "```",
    "",
    "### Random Forest",
    "",
    "```python",
    "RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)",
    "```",
    "",
    "### MLPClassifier",
    "",
    "```python",
    "Pipeline([StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', alpha=0.0005, batch_size=64, learning_rate_init=0.001, max_iter=500, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=42)])",
    "```",
    "",
    "## Results",
    "",
    "| Model | Role | CV F1 mean | CV F1 std | CV ROC AUC mean | Test F1 | Test ROC AUC | Test balanced accuracy | Test precision | Test recall |",
    "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|",
]
for row in comparison_rows:
    report_lines.append(
        f"| {row['Model']} | {row['Role']} | {format_metric(row['CV F1 mean'])} | {format_metric(row['CV F1 std'])} | {format_metric(row['CV ROC AUC mean'])} | {format_metric(row['Test F1'])} | {format_metric(row['Test ROC AUC'])} | {format_metric(row['Test balanced accuracy'])} | {format_metric(row['Test precision'])} | {format_metric(row['Test recall'])} |"
    )
report_lines.extend([
    "",
    "## Champion selection",
    "",
    f"**Selected model: {champion_name}.** The baseline and both candidate models were evaluated with the same data split and validation protocol. The champion is selected first by cross-validated F1, then by CV ROC AUC and held-out test F1. The serialized champion is `models/{champion_path.name}`.",
    "",
    f"The Logistic Regression MVP baseline is retained as `models/{model_file_names['Logistic Regression (MVP baseline)']}` for reproducibility. The non-champion candidate is also persisted under `models/`.",
    "",
])
report_path = results_dir / "phase-2-model-comparison.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"Wrote report: {report_path}")


Wrote report: /Users/falcon/Developer/FIAP-mle/tech-challenge-fase1/docs/results/phase-2-model-comparison.md
